In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visualization style for all plots
sns.set_theme(style="whitegrid", palette="muted")

# Define the directory where your 17 CSV files are stored
DATA_DIR = '../data/PRO-ACT_Data/2026_02_27_PROACT_ALL_FORMS/' 

# ---------------------------------------------------------
# 1. DATA LOADING
# ---------------------------------------------------------
def load_data(filename):
    """Safely load a CSV file into a pandas DataFrame."""
    file_path = os.path.join(DATA_DIR, filename)
    try:
        print(f"Loading {filename}...")
        return pd.read_csv(file_path, low_memory=False)
    except FileNotFoundError:
        print(f"  -> Warning: {filename} not found. Skipping.")
        return None

print("=== PART 1: LOADING DATASETS ===")
datasets = {
    'demographics': load_data('F_PROACT_DEMOGRAPHICS.csv'),
    'alsfrs': load_data('F_PROACT_ALSFRS.csv'),
    'death': load_data('F_PROACT_DEATHDATA.csv'),
    'treatment': load_data('F_PROACT_TREATMENT.csv'),
    'riluzole': load_data('F_PROACT_RILUZOLE.csv'),
    'fvc': load_data('F_PROACT_FVC.csv'),
    'svc': load_data('F_PROACT_SVC.csv'),
    'vitals': load_data('F_PROACT_VITALSIGNS.csv'),
    'handgrip': load_data('F_PROACT_HANDGRIPSTRENGTH.csv'),
    'muscle': load_data('F_PROACT_MUSCLESTRENGTH.csv'),
    'neurofilament': load_data('F_PROACT_Neurofilament.csv'),
    'adverse_events': load_data('F_PROACT_ADVERSEEVENTS.csv'),
    'history': load_data('F_PROACT_ALSHISTORY.csv'),
    'conmeds': load_data('F_PROACT_CONMEDS.csv'),
    'elescorial': load_data('F_PROACT_ELESCORIAL.csv'),
    'family': load_data('F_PROACT_FAMILYHISTORY.csv'),
    'labs': load_data('F_PROACT_LABS.csv')
}

=== PART 1: LOADING DATASETS ===
Loading F_PROACT_DEMOGRAPHICS.csv...
Loading F_PROACT_ALSFRS.csv...
Loading F_PROACT_DEATHDATA.csv...
Loading F_PROACT_TREATMENT.csv...
Loading F_PROACT_RILUZOLE.csv...
Loading F_PROACT_FVC.csv...
Loading F_PROACT_SVC.csv...
Loading F_PROACT_VITALSIGNS.csv...
Loading F_PROACT_HANDGRIPSTRENGTH.csv...
Loading F_PROACT_MUSCLESTRENGTH.csv...
Loading F_PROACT_Neurofilament.csv...
Loading F_PROACT_ADVERSEEVENTS.csv...
Loading F_PROACT_ALSHISTORY.csv...
Loading F_PROACT_CONMEDS.csv...
Loading F_PROACT_ELESCORIAL.csv...
Loading F_PROACT_FAMILYHISTORY.csv...
Loading F_PROACT_LABS.csv...


In [9]:
print("=== PART 2: CLEANING ALSFRS-R DATA (SCHEMA CORRECTED) ===")

df_alsfrs = datasets.get('alsfrs').copy()

# 1. Merge the two Q5 columns into a single Q5 column (use Q5a, if missing, use Q5b)
df_alsfrs['Q5_Cutting'] = df_alsfrs['Q5a_Cutting_without_Gastrostomy'].fillna(df_alsfrs['Q5b_Cutting_with_Gastrostomy'])

# 2. Rename the Respiratory columns to match standard 12-item formatting
df_alsfrs = df_alsfrs.rename(columns={
    'R_1_Dyspnea': 'Q10_Dyspnea',
    'R_2_Orthopnea': 'Q11_Orthopnea',
    'R_3_Respiratory_Insufficiency': 'Q12_Respiratory_Insufficiency'
})

# Define the standard PRO-ACT column names for the 12 ALSFRS-R items
alsfrs_r_items = [
    'Q1_Speech', 
    'Q2_Salivation', 
    'Q3_Swallowing', 
    'Q4_Handwriting', 
    'Q5_Cutting', 
    'Q6_Dressing_and_Hygiene', 
    'Q7_Turning_in_Bed', 
    'Q8_Walking', 
    'Q9_Climbing_Stairs', 
    'Q10_Dyspnea', 
    'Q11_Orthopnea', 
    'Q12_Respiratory_Insufficiency'
]

# Define the key identifier columns
id_col = 'subject_id'
time_col = 'ALSFRS_Delta'

# Filter the dataframe to just the columns we need for the model
cols_to_keep = [id_col, time_col] + alsfrs_r_items
df_clean = df_alsfrs[cols_to_keep].copy()

# Force everything to be numeric
for col in cols_to_keep:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Drop any visit where the time, ID, or ANY of the 12 items are missing
initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=cols_to_keep)

# Sort chronologically per patient
df_clean = df_clean.sort_values(by=['subject_id', 'ALSFRS_Delta']).reset_index(drop=True)

print(f"Original ALSFRS records: {len(df_alsfrs)}")
print(f"Cleaned records (Complete 12 items): {len(df_clean)}")
print(f"Total Unique Patients in Cleaned Data: {df_clean['subject_id'].nunique()}")

# Optional: Preview the data to confirm it looks right
display(df_clean.head())

=== PART 2: CLEANING ALSFRS-R DATA (SCHEMA CORRECTED) ===
Original ALSFRS records: 81229
Cleaned records (Complete 12 items): 50630
Total Unique Patients in Cleaned Data: 5717


,subject_id,ALSFRS_Delta,Q1_Speech,Q2_Salivation,Q3_Swallowing,Q4_Handwriting,Q5_Cutting,Q6_Dressing_and_Hygiene,Q7_Turning_in_Bed,Q8_Walking,Q9_Climbing_Stairs,Q10_Dyspnea,Q11_Orthopnea,Q12_Respiratory_Insufficiency
0,3301,5.0,3.0,4.0,3.0,3.0,3.0,2.0,3.0,2.0,1.0,3.0,4.0,4.0
1,3301,40.0,2.0,4.0,3.0,3.0,3.0,2.0,3.0,2.0,1.0,4.0,3.0,4.0
2,3301,98.0,2.0,4.0,3.0,3.0,2.0,2.0,3.0,2.0,1.0,2.0,4.0,4.0
3,3301,161.0,2.0,4.0,3.0,3.0,2.0,2.0,2.0,2.0,1.0,4.0,3.0,4.0
4,3301,221.0,2.0,4.0,3.0,3.0,2.0,2.0,3.0,2.0,1.0,4.0,4.0,4.0


In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Building the Stan Data Pipeline...")

# 1. Start with the complete ALSFRS-R baseline
# Ensure it is sorted by subject and time!
df_model = df_clean.sort_values(['subject_id', 'ALSFRS_Delta']).copy()

# --- EXTRACT AND MERGE COVARIATES ---

# A. Latent Covariates (Time-Invariant: Treatment, Delay, Riluzole)
df_treat = datasets.get('treatment')[['subject_id', 'Treatment_Group_Delta']].rename(columns={'Treatment_Group_Delta': 'Treatment'})
df_ril = datasets.get('riluzole')[['subject_id', 'Subject_used_Riluzole']].rename(columns={'Subject_used_Riluzole': 'Riluzole'})

df_hist = datasets.get('history').copy()
onset_col = [c for c in df_hist.columns if 'onset_delta' in c.lower()][0]
diag_col = [c for c in df_hist.columns if 'diagnosis_delta' in c.lower()][0]
df_hist['Diag_Delay'] = (df_hist[diag_col] - df_hist[onset_col]) / 30.44
df_delay = df_hist[['subject_id', 'Diag_Delay']].drop_duplicates('subject_id')

# Merge static covariates
df_model = df_model.merge(df_treat.drop_duplicates('subject_id'), on='subject_id', how='left')
df_model = df_model.merge(df_ril.drop_duplicates('subject_id'), on='subject_id', how='left')
df_model = df_model.merge(df_delay, on='subject_id', how='left')

# B. Measurement Covariates (Time-Varying: Uric Acid)
df_labs = datasets.get('labs')
df_uric = df_labs[df_labs['Test_Name'].astype(str).str.contains('Uric Acid', case=False, na=False)].copy()
df_uric['Laboratory_Delta'] = pd.to_numeric(df_uric['Laboratory_Delta'], errors='coerce')
df_uric['Uric_Acid'] = pd.to_numeric(df_uric['Test_Result'], errors='coerce')

# Drop missing values and SORT the right side by time
df_uric = df_uric.dropna(subset=['subject_id', 'Laboratory_Delta', 'Uric_Acid']).sort_values('Laboratory_Delta')

# 1. Force exact types for merge_asof
df_model['subject_id'] = df_model['subject_id'].astype('int64')
df_uric['subject_id'] = df_uric['subject_id'].astype('int64')
df_model['ALSFRS_Delta'] = df_model['ALSFRS_Delta'].astype('float64')

# 2. *** THE CRITICAL FIX ***
# Explicitly sort both dataframes by their time columns immediately before the merge
df_model = df_model.sort_values('ALSFRS_Delta')
df_uric = df_uric.sort_values('Laboratory_Delta')

# 3. Fuzzy merge Uric Acid (within 14 days)
df_model = pd.merge_asof(
    df_model, 
    df_uric[['subject_id', 'Laboratory_Delta', 'Uric_Acid']],
    left_on='ALSFRS_Delta',
    right_on='Laboratory_Delta',
    by='subject_id',
    direction='nearest',
    tolerance=14
)

# 4. Sort back by Patient -> Time so it is perfectly formatted for the Stan model
df_model = df_model.sort_values(['subject_id', 'ALSFRS_Delta']).reset_index(drop=True)

# --- IMPUTATION AND NORMALIZATION ---
# Stan fails if any element in the X matrix is NaN.

# Categorical mapping & imputation (0 for missing/placebo/no, 1 for active/yes)
df_model['Treatment'] = df_model['Treatment'].astype(str).apply(lambda x: 1 if 'Active' in x else 0)
df_model['Riluzole'] = df_model['Riluzole'].astype(str).apply(lambda x: 1 if 'Yes' in x else 0)

# Continuous imputation: Forward fill within patient, then median for total missing
df_model['Uric_Acid'] = df_model.groupby('subject_id')['Uric_Acid'].ffill()
df_model['Uric_Acid'] = df_model['Uric_Acid'].fillna(df_model['Uric_Acid'].median())
df_model['Diag_Delay'] = df_model['Diag_Delay'].fillna(df_model['Diag_Delay'].median())

# Standard Scaling (Z-score) for continuous predictors
scaler = StandardScaler()
df_model[['Diag_Delay', 'Uric_Acid']] = scaler.fit_transform(df_model[['Diag_Delay', 'Uric_Acid']])

# --- STAN FORMATTING ---

# Create contiguous Subject IDs (1 to Nsub)
unique_subs = df_model['subject_id'].unique()
sub_map = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_subs)}
df_model['stan_id'] = df_model['subject_id'].map(sub_map)

# Calculate deltat (Time in months since last visit. 0 for first visit)
df_model['deltat'] = df_model.groupby('stan_id')['ALSFRS_Delta'].diff().fillna(0) / 30.44

# Grouping indices
repme_series = df_model.groupby('stan_id').size()
repme = repme_series.values
cumu = repme_series.cumsum().values

# Y Matrix: Items Q1 to Q12. 
# Stan ordered_logistic requires categories to start at 1. Since ALSFRS is 0-4, we add 1 to make it 1-5.
y_cols = ['Q1_Speech', 'Q2_Salivation', 'Q3_Swallowing', 
          'Q4_Handwriting', 'Q5_Cutting', 'Q6_Dressing_and_Hygiene', 
          'Q7_Turning_in_Bed', 'Q8_Walking', 'Q9_Climbing_Stairs', 
          'Q10_Dyspnea', 'Q11_Orthopnea', 'Q12_Respiratory_Insufficiency']

Y_mat = df_model[y_cols].values.astype(int) + 1 

# Predictor Matrices
X_dyn = df_model[['Treatment', 'Diag_Delay', 'Riluzole']].values
X_meas = df_model[['Uric_Acid']].values

# The final dictionary to feed into Stan
stan_data = {
    'N': len(df_model),
    'Nsub': len(unique_subs),
    'K': 12,
    'R': 4,
    'p_dyn': 3,
    'p_meas': 1,
    'ID': df_model['stan_id'].values.astype(int),
    'cumu': cumu.astype(int),
    'repme': repme.astype(int),
    'Y': Y_mat,
    'deltat': df_model['deltat'].values,
    'X_dyn': X_dyn,
    'X_meas': X_meas
}

print("Stan data dictionary perfectly formatted!")

Building the Stan Data Pipeline...
Stan data dictionary perfectly formatted!


SyntaxError: unterminated string literal (detected at line 2) (2058129017.py, line 2)

In [ ]:
import os
import pandas as pd
from cmdstanpy import CmdStanModel

# ==========================================
# 1. OPTIONAL: SUBSET FOR TESTING
# ==========================================
# Set this to True to test on 100 patients first. Set to False for the final run.
TEST_RUN = True

if TEST_RUN:
    print("WARNING: Running in TEST mode (100 patients).")
    test_subjects = unique_subs[:100]
    df_model_run = df_model[df_model['subject_id'].isin(test_subjects)].copy()
    
    # Re-calculate stan_data indices for the subset
    unique_subs_run = df_model_run['subject_id'].unique()
    sub_map_run = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_subs_run)}
    df_model_run['stan_id'] = df_model_run['subject_id'].map(sub_map_run)
    
    repme_series = df_model_run.groupby('stan_id').size()
    repme_run = repme_series.values
    cumu_run = repme_series.cumsum().values
    
    Y_mat_run = df_model_run[y_cols].values.astype(int) + 1 
    X_dyn_run = df_model_run[['Treatment', 'Diag_Delay', 'Riluzole']].values
    X_meas_run = df_model_run[['Uric_Acid']].values
    
    run_data = {
        'N': len(df_model_run),
        'Nsub': len(unique_subs_run),
        'K': 12,
        'R': 4,
        'p_dyn': 3,
        'p_meas': 1,
        'ID': df_model_run['stan_id'].values.astype(int),
        'cumu': cumu_run.astype(int),
        'repme': repme_run.astype(int),
        'Y': Y_mat_run,
        'deltat': df_model_run['deltat'].values,
        'X_dyn': X_dyn_run,
        'X_meas': X_meas_run
    }
else:
    print("Running FULL model on all patients.")
    run_data = stan_data # Use the dictionary we built in the previous step

# ==========================================
# 2. COMPILE THE STAN MODEL
# ==========================================
stan_file_path = 'lou_alsfrs_12item.stan'

# (Assuming you saved the Stan code from the previous step into 'lou_alsfrs_12item.stan')
if not os.path.exists(stan_file_path):
    raise FileNotFoundError(f"Could not find {stan_file_path}. Please save the Stan code to this file first.")

print(f"\nCompiling {stan_file_path}...")
print("This may take a minute as C++ code is being generated and compiled.")
model = CmdStanModel(stan_file=stan_file_path)
print("Compilation successful!")

# ==========================================
# 3. RUN MCMC SAMPLING
# ==========================================
print("\nStarting MCMC Sampling...")
print("Parameters configured for complex hierarchical geometry (high adapt_delta).")

# Fit the model
# Using 4 chains in parallel. Adjust parallel_chains to match your CPU cores.
fit = model.sample(
    data=run_data,
    chains=4,
    parallel_chains=4, 
    iter_warmup=1000,    # Warmup iterations per chain
    iter_sampling=1000,  # Recorded iterations per chain
    adapt_delta=0.95,    # Increased from default 0.8 to handle the complex LOU transitions
    max_treedepth=12,    # Increased from default 10 to prevent premature trajectory stops
    show_progress=True   # Shows the progress bar in your terminal/notebook
)

print("\nSampling Complete!")

# ==========================================
# 4. DIAGNOSTICS AND RESULTS SUMMARY
# ==========================================
# Run built-in diagnostics to check for divergent transitions or low Bulk-ESS
print("\n=== HMC DIAGNOSTICS ===")
print(fit.diagnose())

print("\n=== EXTRACTING SUMMARY ===")
# Generate a summary dataframe for the parameters of interest
# We exclude the massive random effect matrices (b_raw, xi_raw, xi) from the printout to save memory
summary_df = fit.summary()

# Look specifically at the covariate effects
print("\nDynamic Covariate Effects (alpha_dyn):")
# alpha_dyn[1,1] = Effect of Treatment on Bulbar state
# alpha_dyn[2,2] = Effect of Delay on Fine Motor state, etc.
alpha_summary = summary_df.loc[summary_df.index.str.startswith('alpha_dyn')]
display(alpha_summary[['Mean', 'MCSE', 'StdDev', '5%', '95%', 'R_hat']])

print("\nMeasurement Covariate Effects (beta - Uric Acid):")
beta_summary = summary_df.loc[summary_df.index.str.startswith('beta')]
display(beta_summary[['Mean', 'MCSE', 'StdDev', '5%', '95%', 'R_hat']])

# Save the full summary to CSV for your records
summary_df.to_csv('lou_model_summary.csv')
print("\nFull parameter summary saved to 'lou_model_summary.csv'.")

In [12]:
import os
import pandas as pd
from cmdstanpy import CmdStanModel

print("=== PART 3: STAN MODEL INTEGRATION ===")

# ---------------------------------------------------------
# 1. WRITE THE STAN CODE TO A FILE
# ---------------------------------------------------------
stan_file_path = 'lou_alsfrs_12item.stan'

# This is the exact Stan code we built previously
stan_code = """
functions {
  matrix solve_lyapunov(matrix Gamma, matrix Sigma, int R) {
    matrix[R*R, R*R] K;
    vector[R*R] vec_Sigma;
    vector[R*R] vec_Omega;
    matrix[R, R] Omega;
    
    for (i in 1:R) {
      for (j in 1:R) {
        for (k in 1:R) {
          for (l in 1:R) {
            int row_idx = (j - 1) * R + i;
            int col_idx = (l - 1) * R + k;
            K[row_idx, col_idx] = (i == k ? Gamma[j, l] : 0.0) + (j == l ? Gamma[i, k] : 0.0);
          }
        }
      }
    }
    
    for (i in 1:R) {
      for (j in 1:R) {
        vec_Sigma[(j - 1) * R + i] = Sigma[i, j];
      }
    }
    
    vec_Omega = K \\ vec_Sigma;
    
    for (i in 1:R) {
      for (j in 1:R) {
        Omega[i, j] = vec_Omega[(j - 1) * R + i];
      }
    }
    return Omega;
  }
}

data {
    int<lower=1> N;         
    int<lower=1> Nsub;      
    int<lower=1> K;         
    int<lower=1> R;         
    int<lower=1> p_dyn;     
    int<lower=1> p_meas;    

    array[N] int<lower=1, upper=Nsub> ID;
    array[Nsub] int cumu;
    array[Nsub] int repme;
    
    array[N, K] int<lower=1, upper=5> Y;

    vector[N] deltat;
    matrix[N, p_dyn] X_dyn;
    matrix[N, p_meas] X_meas;
}

parameters {
    array[K] ordered[4] theta; 

    vector<lower=1e-6>[K - R] lambda_free;
    real<lower=1e-6> sigma_lambda;

    matrix[K, p_meas] beta;       
    matrix[R, p_dyn] alpha_dyn;   

    matrix[Nsub, K] b_raw;
    vector<lower=1e-6>[K] sigma_bk;

    cholesky_factor_corr[R] L_S_corr;     
    vector<lower=0>[R] L_S_scale;         
    
    vector[R * (R - 1) / 2] a_low;
    
    cholesky_factor_corr[R] L_Sigma_corr; 
    vector<lower=0>[R] L_Sigma_scale;     
    
    matrix[R, N] xi_raw;
}

transformed parameters {
    matrix[Nsub, K] b;
    vector[K] lambda; 
    
    matrix[R, R] L_S;
    matrix[R, R] S;
    matrix[R, R] A;
    matrix[R, R] Gamma;
    
    matrix[R, R] L_Sigma;
    matrix[R, R] Sigma;
    matrix[R, R] Omega;
    matrix[R, N] xi;

    lambda[1] = 1.0;              
    lambda[2] = lambda_free[1];
    lambda[3] = lambda_free[2];
    
    lambda[4] = 1.0;              
    lambda[5] = lambda_free[3];
    lambda[6] = lambda_free[4];
    
    lambda[7] = 1.0;              
    lambda[8] = lambda_free[5];
    lambda[9] = lambda_free[6];
    
    lambda[10] = 1.0;              
    lambda[11] = lambda_free[7];
    lambda[12] = lambda_free[8];

    for (i in 1:Nsub){
        for (k in 1:K){
            b[i, k] = b_raw[i, k] * sigma_bk[k];
        }
    }
    
    L_S = diag_pre_multiply(L_S_scale, L_S_corr);
    S = multiply_lower_tri_self_transpose(L_S);
    
    A = rep_matrix(0, R, R);
    {
      int pos = 1;
      for (i in 2:R) {
        for (j in 1:(i-1)) {
          A[i, j] = a_low[pos];
          A[j, i] = -a_low[pos];
          pos += 1;
        }
      }
    }
    Gamma = S + A;

    L_Sigma = diag_pre_multiply(L_Sigma_scale, L_Sigma_corr);
    Sigma = multiply_lower_tri_self_transpose(L_Sigma);
    
    Omega = solve_lyapunov(Gamma, Sigma, R);
    Omega = 0.5 * (Omega + Omega'); 
    Omega = add_diag(Omega, 1e-5);

    {
        matrix[R, R] L_Omega = cholesky_decompose(Omega);
        for (i in 1:Nsub) {
            int start_idx = cumu[i] - repme[i] + 1;
            
            vector[R] mu_start = alpha_dyn * X_dyn[start_idx, ]';
            xi[:, start_idx] = mu_start + L_Omega * xi_raw[:, start_idx];
            
            for (j in 2:repme[i]) {
                int k = start_idx + j - 1;
                matrix[R, R] Phi = matrix_exp(-deltat[k] * Gamma);
                
                matrix[R, R] Q = Omega - Phi * Omega * Phi';
                matrix[R, R] Q_sym = 0.5 * (Q + Q'); 
                matrix[R, R] L_Q = cholesky_decompose(add_diag(Q_sym, 1e-6));
                
                vector[R] mu_k = alpha_dyn * X_dyn[k, ]';
                vector[R] mu_prev = alpha_dyn * X_dyn[k-1, ]';
                
                xi[:, k] = mu_k + Phi * (xi[:, k-1] - mu_prev) + L_Q * xi_raw[:, k];
            }
        }
    }
}

model {
    for (k in 1:K) {
        theta[k] ~ normal(0, 5); 
    }
    
    lambda_free ~ normal(1, sigma_lambda);
    sigma_lambda ~ cauchy(0, 5);
    
    to_vector(alpha_dyn) ~ normal(0, 5);
    to_vector(beta) ~ cauchy(0, 5);
    
    sigma_bk ~ cauchy(0, 5);
    to_vector(b_raw) ~ normal(0, 1);
    
    L_S_corr ~ lkj_corr_cholesky(2.0);
    L_S_scale ~ lognormal(0, 0.5); 
    
    a_low ~ normal(0, 0.5);
    
    L_Sigma_corr ~ lkj_corr_cholesky(2.0);
    L_Sigma_scale ~ lognormal(0, 0.5);
    
    to_vector(xi_raw) ~ std_normal();

    for (i in 1:N) {
        int sub = ID[i];
        row_vector[p_meas] Xi_m = X_meas[i, ];
        
        Y[i, 1] ~ ordered_logistic(Xi_m * beta[1, ]' + lambda[1] * xi[1, i] + b[sub, 1], theta[1]);
        Y[i, 2] ~ ordered_logistic(Xi_m * beta[2, ]' + lambda[2] * xi[1, i] + b[sub, 2], theta[2]);
        Y[i, 3] ~ ordered_logistic(Xi_m * beta[3, ]' + lambda[3] * xi[1, i] + b[sub, 3], theta[3]);
        
        Y[i, 4] ~ ordered_logistic(Xi_m * beta[4, ]' + lambda[4] * xi[2, i] + b[sub, 4], theta[4]);
        Y[i, 5] ~ ordered_logistic(Xi_m * beta[5, ]' + lambda[5] * xi[2, i] + b[sub, 5], theta[5]);
        Y[i, 6] ~ ordered_logistic(Xi_m * beta[6, ]' + lambda[6] * xi[2, i] + b[sub, 6], theta[6]);
        
        Y[i, 7] ~ ordered_logistic(Xi_m * beta[7, ]' + lambda[7] * xi[3, i] + b[sub, 7], theta[7]);
        Y[i, 8] ~ ordered_logistic(Xi_m * beta[8, ]' + lambda[8] * xi[3, i] + b[sub, 8], theta[8]);
        Y[i, 9] ~ ordered_logistic(Xi_m * beta[9, ]' + lambda[9] * xi[3, i] + b[sub, 9], theta[9]);
        
        Y[i, 10] ~ ordered_logistic(Xi_m * beta[10, ]' + lambda[10] * xi[4, i] + b[sub, 10], theta[10]);
        Y[i, 11] ~ ordered_logistic(Xi_m * beta[11, ]' + lambda[11] * xi[4, i] + b[sub, 11], theta[11]);
        Y[i, 12] ~ ordered_logistic(Xi_m * beta[12, ]' + lambda[12] * xi[4, i] + b[sub, 12], theta[12]);
    }
}
"""

# Write the string to a physical file
with open(stan_file_path, 'w') as f:
    f.write(stan_code)
print(f"Successfully wrote Stan code to {stan_file_path}")

# ---------------------------------------------------------
# 2. COMPILE THE MODEL
# ---------------------------------------------------------
print("\nCompiling model (this may take 1-2 minutes as C++ builds)...")
model = CmdStanModel(stan_file=stan_file_path)
print("Compilation successful!")

# ---------------------------------------------------------
# 3. RUN A SMALL TEST BATCH
# ---------------------------------------------------------
# We will test on just 50 patients first to ensure the math doesn't crash
print("\nPreparing test batch (50 patients)...")
test_subjects = unique_subs[:50]
df_model_run = df_model[df_model['subject_id'].isin(test_subjects)].copy()

# Re-map IDs for the subset
unique_subs_run = df_model_run['subject_id'].unique()
sub_map_run = {old_id: new_id + 1 for new_id, old_id in enumerate(unique_subs_run)}
df_model_run['stan_id'] = df_model_run['subject_id'].map(sub_map_run)

repme_series = df_model_run.groupby('stan_id').size()
repme_run = repme_series.values
cumu_run = repme_series.cumsum().values

run_data = {
    'N': len(df_model_run),
    'Nsub': len(unique_subs_run),
    'K': 12,
    'R': 4,
    'p_dyn': 3,
    'p_meas': 1,
    'ID': df_model_run['stan_id'].values.astype(int),
    'cumu': cumu_run.astype(int),
    'repme': repme_run.astype(int),
    'Y': df_model_run[y_cols].values.astype(int) + 1,
    'deltat': df_model_run['deltat'].values,
    'X_dyn': df_model_run[['Treatment', 'Diag_Delay', 'Riluzole']].values,
    'X_meas': df_model_run[['Uric_Acid']].values
}

print("Starting MCMC Sampling on test batch...")
fit = model.sample(
    data=run_data,
    chains=4,
    parallel_chains=4, 
    iter_warmup=500,    # Reduced for the test run
    iter_sampling=500,  # Reduced for the test run
    adapt_delta=0.95,   
    max_treedepth=12,   
    show_progress=True  
)

print("\nSampling Complete! Here is a summary of the covariate effects:")
summary_df = fit.summary()

# Display the Alpha (Latent dynamics) and Beta (Measurement) parameters
display(summary_df.loc[summary_df.index.str.startswith('alpha_dyn')])
display(summary_df.loc[summary_df.index.str.startswith('beta')])

/u/zwu1/.conda/envs/ou/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
19:48:55 - cmdstanpy - INFO - compiling stan file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/lou_alsfrs_12item.stan to exe file /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/lou_alsfrs_12item


=== PART 3: STAN MODEL INTEGRATION ===
Successfully wrote Stan code to lou_alsfrs_12item.stan

Compiling model (this may take 1-2 minutes as C++ builds)...


19:50:15 - cmdstanpy - INFO - compiled model executable: /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/lou_alsfrs_12item
19:50:15 - cmdstanpy - WARNING - Stan compiler has produced 1 warnings:
19:50:15 - cmdstanpy - WARNING - 
--- Translating Stan model to C++ code ---
bin/stanc --filename-in-msg=lou_alsfrs_12item.stan --o=/nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/lou_alsfrs_12item.hpp /nfs/nfs9/home/nobackup/zwu1/gnpc_explor/src/lou_alsfrs_12item.stan
Warning in 'lou_alsfrs_12item.stan', line 72, column 11 to column 22:
    Found int division:
        R * (R - 1) / 2
    Values will be rounded towards zero. If rounding is not desired you can
    write the division as
        R * (R - 1) / 2.0
    If rounding is intended please use the integer division operator %/%.

--- Compiling C++ code ---
g++ -std=c++17 -pthread -D_REENTRANT -Wno-sign-compare -Wno-ignored-attributes -Wno-class-memaccess      -I stan/lib/stan_math/lib/tbb_2020.3/include    -O3 -I src -I stan/src -I stan/lib/rapid

Compilation successful!

Preparing test batch (50 patients)...
Starting MCMC Sampling on test batch...


chain 1:   0%|          | 0/1000 [00:00<?, ?it/s, (Warmup)]





chain 1:   0%|          | 1/1000 [00:00<05:54,  2.82it/s, (Warmup)]

chain 1:  10%|█         | 100/1000 [1:11:46<10:47:50, 43.19s/it, (Warmup)]


chain 1:  20%|██        | 200/1000 [1:31:30<5:29:44, 24.73s/it, (Warmup)] 


chain 1:  30%|███       | 300/1000 [1:45:21<3:20:59, 17.23s/it, (Warmup)]


chain 1:  40%|████      | 400/1000 [1:54:03<2:04:49, 12.48s/it, (Warmup)]






chain 1:  50%|█████     | 501/1000 [2:04:23<1:24:40, 10.18s/it, (Sampling)]

chain 1:  60%|██████    | 600/1000 [2:12:41<53:25,  8.01s/it, (Sampling)]  




chain 1:  70%|███████   | 700/1000 [2:21:03<34:25,  6.89s/it, (Sampling)]


chain 1:  80%|████████  | 800/1000 [2:29:27<20:47,  6.24s/it, (Sampling)]


chain 1:  90%|█████████ | 900/1000 [2:37:50<09:43,  5.84s/it, (Sampling)]


chain 1: 100%|██████████| 1000/1000 [2:46:15<00:00,  5.58s/it, (Sampling)]

chain 2: 100%|██████████| 1000/1000 [3:07:25<00:00, 11.25s/it, (Sampling completed)]

chain 3: 


22:57:40 - cmdstanpy - INFO - CmdStan done processing.
22:57:40 - cmdstanpy - WARNING - Non-fatal error during sampling:
Exception: cholesky_decompose: A is not symmetric. A[1,2] = nan, but A[2,1] = nan (in 'lou_alsfrs_12item.stan', line 140, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = nan, but A[2,1] = nan (in 'lou_alsfrs_12item.stan', line 140, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = nan, but A[2,1] = nan (in 'lou_alsfrs_12item.stan', line 140, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'lou_alsfrs_12item.stan', line 140, column 8 to column 57)
	Exception: cholesky_decompose: A is not symmetric. A[1,2] = -nan, but A[2,1] = -nan (in 'lou_alsfrs_12item.stan', line 140, column 8 to column 57)
	Exception: cholesky_decompose: Matrix m is not positive definite (in 'lou_alsfrs_12item.stan', line 153, column 16 to column 77)
	Exception: cho

22:57:41 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 3 had 1 divergent transitions (0.2%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.



Sampling Complete! Here is a summary of the covariate effects:


,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
"alpha_dyn[1,1]",-0.050618,0.074809,4.86146,4.618840,-8.298600,0.027356,8.414790,4403.960,1440.880,0.478430,1.00235
"alpha_dyn[1,2]",1.373730,0.086064,1.61780,1.622750,-1.276840,1.385250,3.898340,350.462,743.523,0.038073,1.01796
"alpha_dyn[1,3]",3.375330,0.140990,2.23253,2.147790,-0.255741,3.309230,7.068330,268.225,786.583,0.029139,1.01252
"alpha_dyn[2,1]",0.009075,0.080039,5.07869,5.226580,-8.493370,0.074494,8.441720,4101.490,1540.290,0.445571,1.00400
"alpha_dyn[2,2]",-1.080620,0.041606,1.01172,0.996489,-2.706340,-1.091340,0.597913,587.923,1076.480,0.063870,1.00402
"alpha_dyn[2,3]",0.909133,0.062930,1.43327,1.429670,-1.542840,0.898520,3.289270,519.693,850.162,0.056458,1.00658
"alpha_dyn[3,1]",-0.081870,0.078559,5.09130,5.075830,-8.465510,-0.074136,8.449630,4035.630,1163.830,0.438415,1.00275
"alpha_dyn[3,2]",-0.183668,0.043778,1.10442,1.076500,-1.964420,-0.217166,1.607830,642.361,1201.280,0.069784,1.00519
"alpha_dyn[3,3]",2.126690,0.071360,1.63702,1.559470,-0.671662,2.139600,4.818500,531.807,812.102,0.057773,1.00934
"alpha_dyn[4,1]",0.203592,0.082473,4.98701,5.003350,-8.097760,0.152731,8.608080,3612.380,1327.760,0.392435,1.00387


,Mean,MCSE,StdDev,MAD,5%,50%,95%,ESS_bulk,ESS_tail,ESS_bulk/s,R_hat
"beta[1,1]",1.789280,0.023105,0.743559,0.746178,0.609254,1.774830,3.015480,1088.090,1216.06,0.118206,1.001600
"beta[2,1]",0.412990,0.014580,0.473237,0.478941,-0.386218,0.401006,1.189220,1075.270,1369.80,0.116814,0.999683
"beta[3,1]",1.256170,0.027058,0.740274,0.727993,0.064108,1.232300,2.530060,772.598,1218.87,0.083932,1.002650
"beta[4,1]",0.342544,0.014421,0.462864,0.451739,-0.401093,0.345938,1.097080,1064.460,1151.72,0.115639,1.003440
"beta[5,1]",-0.241344,0.021225,0.653057,0.654080,-1.300350,-0.246659,0.836241,965.741,1309.91,0.104915,1.008350
"beta[6,1]",-0.575036,0.010315,0.363485,0.361371,-1.168430,-0.576618,0.012063,1251.440,1468.48,0.135952,1.000680
"beta[7,1]",-0.341481,0.015902,0.555440,0.554121,-1.264950,-0.327132,0.533855,1355.890,1377.48,0.147298,1.003650
"beta[8,1]",0.763701,0.015729,0.595418,0.595978,-0.173022,0.751100,1.792110,1457.990,1572.47,0.158390,1.002120
"beta[9,1]",0.206221,0.018363,0.587854,0.595017,-0.752969,0.201245,1.178200,1045.690,1498.46,0.113600,1.003290
"beta[10,1]",0.245953,0.015692,0.611104,0.581507,-0.740446,0.226827,1.257000,1535.660,1617.34,0.166828,1.000790
